# Zambia Smallholder Irrigation — Phase 3 (Extensions, Real Sentinel-2 Data)

**Reproducibility study of *“Mapping small-scale irrigation for climate adaptation.”***
This single notebook runs the entire Phase 3 pipeline **end-to-end on the real Sentinel-2
stacks** (no simulation):

1. **Download** the Sentinel-2 stacks from Kaggle (part 1 + part 2)
2. **Load** the real labels (`latest_irrigation_table.csv`)
3. **Build features** from the real `*_stack.tif` time series (NDVI / NDWI / EVI trajectories)
4. **Train** the reference Random-Forest baseline + a zoo of lightweight models
5. **Phase 3 extensions**, each with figures + tables:
   - 3a Lightweight models — accuracy vs model-size vs inference latency
   - 3b Robustness — cloud-gap & sensor-noise degradation + cross-year transfer
   - 3c Cross-region generalisation — leave-one-province-out
   - 3d Fairness — smallholder vs large-farm detection (equal-opportunity)
   - 3e Explainability — feature importance + SHAP

Runs unchanged on **Kaggle** (attach both datasets) or **locally** (uses the Kaggle API to
download). Every output is real-data; nothing here is simulated.


## 0 · Setup

In [2]:
import sys, subprocess
def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
# rasterio for reading the stacks; shap for XAI. (Kaggle already has most of these.)
for p in ["rasterio", "scikit-learn", "shap", "seaborn", "kaggle"]:
    try:
        __import__(p if p != "scikit-learn" else "sklearn")
    except Exception:
        pip(p)

import os, glob, json, time, pickle, warnings, math
from pathlib import Path
import numpy as np, pandas as pd
import rasterio
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")
RNG = 42
ON_KAGGLE = os.path.exists("/kaggle/input")
OUT = Path("outputs/phase3"); OUT.mkdir(parents=True, exist_ok=True)
PALETTE = sns.color_palette("viridis", 8)
print("Running on Kaggle:" , ON_KAGGLE)


Running on Kaggle: False


## 1 · Get the Sentinel-2 stacks

- **On Kaggle:** attach both datasets via *Add Input* — they appear under `/kaggle/input/`.
- **Locally:** needs a Kaggle API token at `~/.kaggle/kaggle.json`
  (Kaggle → Settings → API → *Create New API Token*). The cell downloads + unzips both parts.


In [3]:
DATASETS = [
    "wenhaolu49/zambia-smallholder-irrigation-sentinel2-part1",
    "wenhaolu49/zambia-smallholder-irrigation-sentinel2-part2",
]
SEARCH_ROOTS = []
if ON_KAGGLE:
    SEARCH_ROOTS = glob.glob("/kaggle/input/*")
else:
    dst = Path("data/features/_kaggle"); dst.mkdir(parents=True, exist_ok=True)
    for ds in DATASETS:
        slug = ds.split("/")[1]
        if not list(dst.glob(f"{slug}*")) and not list((dst).rglob("*_stack*.tif")):
            print("Downloading", ds, "...")
            subprocess.run(["kaggle", "datasets", "download", "-d", ds,
                            "-p", str(dst), "--unzip"], check=True)
    SEARCH_ROOTS = [str(dst)]

# recursively find every masked stack (robust to dataset folder layout)
STACKS = []
for root in SEARCH_ROOTS:
    STACKS += glob.glob(os.path.join(root, "**", "*_stack_masked.tif"), recursive=True)
STACKS = sorted(set(STACKS))
print(f"Found {len(STACKS)} masked stacks across {len(SEARCH_ROOTS)} root(s)")
assert STACKS, "No stacks found — attach the Kaggle datasets (Kaggle) or check kaggle.json (local)."
print("example:", STACKS[0])


Dataset URL: https://www.kaggle.com/datasets/wenhaolu49/zambia-smallholder-irrigation-sentinel2-part1
License(s): MIT


 11%|█▏        | 1.54G/13.5G [02:38<20:39, 10.4MB/s]


User cancelled operation


KeyboardInterrupt: 

## 2 · Load the real labels

In [ ]:
LABEL_CANDIDATES = [
    "data/labels/labeled_surveys/random_sample/latest_irrigation_table.csv",
    "/kaggle/working/latest_irrigation_table.csv",
]
csv_path = next((p for p in LABEL_CANDIDATES if os.path.exists(p)), None)
if csv_path is None:
    import requests
    url = ("https://raw.githubusercontent.com/AIscend-Research/"
           "smallholder-irrigation-dataset/main/data/labels/labeled_surveys/"
           "random_sample/latest_irrigation_table.csv")
    csv_path = "latest_irrigation_table.csv"
    open(csv_path, "wb").write(requests.get(url, timeout=60).content)
labels = pd.read_csv(csv_path)
print("labels:", labels.shape)

# file_id used by the downloader:  {site_id w/o 'id_'}_{YYYY.MM.DD}
def row_file_id(r):
    sid = str(r["site_id"]).replace("id_", "")
    return f"{sid}_{int(r['year'])}.{int(r['month']):02d}.{int(r['day']):02d}"
labels["file_id"] = labels.apply(row_file_id, axis=1)
labels["target"]  = (labels["irrigation"] >= 2).astype(int)
labels.head(3)


## 3 · Build features from the real stacks

Each masked stack is `(420, 100, 100)` = 10 bands × 42 decadal windows (band-major:
layer = `band*42 + window`). For every window we compute **NDVI, NDWI, EVI** per pixel
(0 = nodata/cloud), then aggregate over the ~1 km AOI to per-window summaries:

- `ndvi_mean`, `ndvi_p90` (90th pct — preserves small bright-green irrigated patches),
- `ndwi_mean`, `evi_mean`.

That yields four length-42 **trajectories** per image. Keeping the trajectories (not just
scalars) lets us re-derive features under simulated cloud loss for the robustness test.


In [ ]:
S2_BANDS = ["B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12"]
BIDX = {b:i for i,b in enumerate(S2_BANDS)}
NWIN = 42

def _band(stack, b, t):           # band-major: layer = band*42 + window
    return stack[BIDX[b]*NWIN + t].astype(np.float32)

def stack_trajectories(path):
    with rasterio.open(path) as src:
        arr = src.read()                      # (420, H, W) uint16
    H, W = arr.shape[1], arr.shape[2]
    nd_mean = np.full(NWIN, np.nan); nd_p90 = np.full(NWIN, np.nan)
    nw_mean = np.full(NWIN, np.nan); ev_mean = np.full(NWIN, np.nan)
    for t in range(NWIN):
        red = _band(arr,"B4",t); nir = _band(arr,"B8",t)
        green = _band(arr,"B3",t); swir = _band(arr,"B11",t); blue = _band(arr,"B2",t)
        valid = (red>0)&(nir>0)                # masked/missing pixels are 0
        if valid.sum() < 20:                   # window essentially missing
            continue
        r=red[valid]; n=nir[valid]; sw=swir[valid]; bl=blue[valid]
        ndvi = (n-r)/(n+r+1e-6)
        ndwi = (n-sw)/(n+sw+1e-6)
        nirs, reds, blues = n/10000., r/10000., bl/10000.
        evi = 2.5*(nirs-reds)/(nirs + 6*reds - 7.5*blues + 1 + 1e-6)
        nd_mean[t]=np.nanmean(ndvi); nd_p90[t]=np.nanpercentile(ndvi,90)
        nw_mean[t]=np.nanmean(ndwi); ev_mean[t]=np.nanmean(evi)
    return dict(ndvi=nd_mean, ndvi_p90=nd_p90, ndwi=nw_mean, evi=ev_mean)

# map available stacks by file_id
stack_by_id = {}
for p in STACKS:
    fid = os.path.basename(p).replace("_stack_masked.tif","")
    stack_by_id[fid] = p
labels_avail = labels[labels["file_id"].isin(stack_by_id)].reset_index(drop=True)
print(f"labelled images with a stack: {len(labels_avail)} / {len(labels)}")

CACHE = OUT / "real_trajectories.npz"
if CACHE.exists():
    z = np.load(CACHE, allow_pickle=True)
    TRAJ = {k: z[k] for k in ["ndvi","ndvi_p90","ndwi","evi"]}
    keep_ids = list(z["file_id"])
    print("loaded cached trajectories:", TRAJ["ndvi"].shape)
else:
    rows_ndvi=[]; rows_p90=[]; rows_ndwi=[]; rows_evi=[]; keep_ids=[]
    t0=time.time()
    for i, fid in enumerate(labels_avail["file_id"]):
        try:
            tr = stack_trajectories(stack_by_id[fid])
        except Exception as e:
            continue
        rows_ndvi.append(tr["ndvi"]); rows_p90.append(tr["ndvi_p90"])
        rows_ndwi.append(tr["ndwi"]); rows_evi.append(tr["evi"]); keep_ids.append(fid)
        if (i+1) % 100 == 0:
            print(f"  {i+1}/{len(labels_avail)}  ({time.time()-t0:.0f}s)")
    TRAJ = dict(ndvi=np.array(rows_ndvi), ndvi_p90=np.array(rows_p90),
                ndwi=np.array(rows_ndwi), evi=np.array(rows_evi))
    np.savez_compressed(CACHE, file_id=np.array(keep_ids), **TRAJ)
    print("built trajectories:", TRAJ["ndvi"].shape, f"in {time.time()-t0:.0f}s")

# align metadata to the images we actually featurized
meta = labels_avail.set_index("file_id").loc[keep_ids].reset_index()


## 4 · Enrich metadata (province from lon/lat, farm-size class)

In [ ]:
PROVINCE_CENTROIDS = {
    "Central":(28.5,-14.5),"Copperbelt":(27.8,-13.0),"Eastern":(32.0,-13.5),
    "Luapula":(29.0,-11.0),"Lusaka":(28.7,-15.4),"Muchinga":(32.0,-11.5),
    "Northern":(31.3,-10.0),"North-Western":(25.0,-13.0),"Southern":(27.0,-16.5),
    "Western":(23.5,-15.0)}
def province(x,y):
    return min(PROVINCE_CENTROIDS, key=lambda k:(x-PROVINCE_CENTROIDS[k][0])**2+(y-PROVINCE_CENTROIDS[k][1])**2)
def farm_class(s):
    if not np.isfinite(s): return "none"
    if s < 500:  return "smallholder"
    if s < 5000: return "medium"
    return "large"
meta["province"]   = [province(x,y) for x,y in zip(meta.x, meta.y)]
meta["farm_class"] = meta["poly_avg_size_hc"].apply(farm_class)
y = meta["target"].to_numpy()
print("images:", len(meta), " positives:", int(y.sum()), f"({y.mean()*100:.1f}%)")
print("farm classes:", meta.farm_class.value_counts().to_dict())
print("provinces:", meta.province.value_counts().to_dict())


## 5 · Featurization + corruption hook

Summarise each trajectory (mean/max/min/std/amplitude/early/late/slope/AUC/#green) and add
cross-index terms. `corrupt()` drops a fraction of windows (cloud) + adds sensor noise for
the robustness experiments — re-deriving features from the trajectories.

In [ ]:
def corrupt(arr, drop_frac, noise, rng):
    out = arr.copy().astype(float); n,T = out.shape
    if drop_frac>0:
        m = rng.random((n,T)) < drop_frac; out[m]=np.nan
    # fill NaNs (existing + dropped) by interpolation along time
    for i in range(n):
        row=out[i]; idx=np.arange(T); good=~np.isnan(row)
        out[i] = np.nanmean(arr) if good.sum()==0 else np.interp(idx, idx[good], row[good])
    if noise>0: out = out + rng.normal(0,noise,out.shape)
    return out

def _sf(a, p):
    T=len(a); late=np.nanmean(a[-3:]); early=np.nanmean(a[:3])
    slope=np.polyfit(np.arange(T),a,1)[0]
    return {f"{p}_mean":np.nanmean(a),f"{p}_max":np.nanmax(a),f"{p}_min":np.nanmin(a),
            f"{p}_std":np.nanstd(a),f"{p}_amp":np.nanmax(a)-np.nanmin(a),
            f"{p}_early":early,f"{p}_late":late,f"{p}_late_minus_early":late-early,
            f"{p}_slope":slope,f"{p}_auc":np.nanmean(a)*T,f"{p}_n_green":int((a>0.4).sum())}

def featurize(traj, drop_frac=0.0, noise=0.0, seed=0):
    rng=np.random.default_rng(seed)
    nd,p90,nw,ev = (traj["ndvi"],traj["ndvi_p90"],traj["ndwi"],traj["evi"])
    if drop_frac>0 or noise>0:
        nd=corrupt(nd,drop_frac,noise,rng); p90=corrupt(p90,drop_frac,noise,rng)
        nw=corrupt(nw,drop_frac,noise,rng); ev=corrupt(ev,drop_frac,noise,rng)
    else:
        nd=corrupt(nd,0,0,rng); p90=corrupt(p90,0,0,rng); nw=corrupt(nw,0,0,rng); ev=corrupt(ev,0,0,rng)
    rows=[]
    for i in range(nd.shape[0]):
        f={}; f.update(_sf(nd[i],"ndvi")); f.update(_sf(p90[i],"ndvip90"))
        f.update(_sf(nw[i],"ndwi")); f.update(_sf(ev[i],"evi"))
        f["ndvi_ndwi_late"]=np.nanmean(nd[i,-3:])*np.nanmean(nw[i,-3:])
        f["p90_minus_mean_late"]=np.nanmean(p90[i,-3:])-np.nanmean(nd[i,-3:])
        for tt in range(0,NWIN,3):  # every 3rd raw window to keep it compact
            f[f"ndvi_t{tt:02d}"]=nd[i,tt]; f[f"ndwi_t{tt:02d}"]=nw[i,tt]
        rows.append(f)
    return pd.DataFrame(rows)

X = featurize(TRAJ)
feat_names = X.columns.tolist()
print("feature matrix:", X.shape)


## 6 · Models + grouped train/test split (no site leakage)

In [ ]:
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
    HistGradientBoostingClassifier, ExtraTreesClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score)
from sklearn.base import clone

def make_models(rs=RNG):
    return {
      "RF-full (baseline)": RandomForestClassifier(n_estimators=400,max_features="sqrt",
                              class_weight="balanced",n_jobs=-1,random_state=rs),
      "RF-small (50, d=8)": RandomForestClassifier(n_estimators=50,max_depth=8,
                              max_features="sqrt",class_weight="balanced",n_jobs=-1,random_state=rs),
      "RF-tiny (15, d=6)":  RandomForestClassifier(n_estimators=15,max_depth=6,
                              max_features="sqrt",class_weight="balanced",n_jobs=-1,random_state=rs),
      "ExtraTrees (50, d=8)":ExtraTreesClassifier(n_estimators=50,max_depth=8,
                              max_features="sqrt",class_weight="balanced",n_jobs=-1,random_state=rs),
      "GradientBoosting":   GradientBoostingClassifier(n_estimators=200,max_depth=3,random_state=rs),
      "HistGB (fast)":      HistGradientBoostingClassifier(max_iter=200,max_depth=6,random_state=rs),
      "DecisionTree (d=6)": DecisionTreeClassifier(max_depth=6,class_weight="balanced",random_state=rs),
      "LogReg":             Pipeline([("s",StandardScaler()),
                              ("c",LogisticRegression(max_iter=2000,class_weight="balanced",random_state=rs))]),
    }

def proba_of(m,Xd):
    if hasattr(m,"predict_proba"): return m.predict_proba(Xd)[:,1]
    s=m.decision_function(Xd); return (s-s.min())/(s.max()-s.min()+1e-9)
def metrics_at(yt,pr,thr=0.5):
    p=(pr>=thr).astype(int)
    return dict(precision=precision_score(yt,p,zero_division=0),
        recall=recall_score(yt,p,zero_division=0),f1=f1_score(yt,p,zero_division=0),
        auprc=average_precision_score(yt,pr),roc_auc=roc_auc_score(yt,pr))

gss=GroupShuffleSplit(n_splits=1,test_size=0.3,random_state=RNG)
tr,te=next(gss.split(X,groups=meta["site_id"]))
Xtr,Xte=X.iloc[tr].reset_index(drop=True),X.iloc[te].reset_index(drop=True)
ytr,yte=y[tr],y[te]; meta_te=meta.iloc[te].reset_index(drop=True)
print(f"train {len(tr)} (pos {ytr.sum()}) / test {len(te)} (pos {yte.sum()})")


## 3a · Lightweight models — accuracy vs size vs latency

In [ ]:
def annotate(ax,fmt="{:.2f}",fs=9):
    for p in ax.patches:
        h=p.get_height()
        if np.isfinite(h): ax.annotate(fmt.format(h),(p.get_x()+p.get_width()/2,h),
                                        ha="center",va="bottom",fontsize=fs)
rows={}; fitted={}
for name,mdl in make_models().items():
    m=clone(mdl); t0=time.perf_counter(); m.fit(Xtr,ytr); tt=time.perf_counter()-t0
    pr=proba_of(m,Xte); met=metrics_at(yte,pr)
    size_kb=len(pickle.dumps(m))/1024
    t1=time.perf_counter(); [proba_of(m,Xte) for _ in range(3)]; ms=((time.perf_counter()-t1)/3/len(Xte))*1e6
    rows[name]=dict(**met,size_kb=size_kb,train_s=tt,infer_ms_per_1k=ms); fitted[name]=m
    print(f"{name:22s} F1={met['f1']:.3f} AUPRC={met['auprc']:.3f} size={size_kb:7.1f}KB lat={ms:6.2f}ms/1k")
dfL=pd.DataFrame(rows).T; dfL.to_csv(OUT/"3a_lightweight_models.csv")

fig,ax=plt.subplots(figsize=(13,6)); dfL[["auprc","f1"]].plot(kind="bar",ax=ax,
    color=[PALETTE[1],PALETTE[4]],width=.8); ax.set_ylim(0,1.05); ax.set_ylabel("Score")
ax.set_title("3a · Detection performance by model (real S2)"); plt.xticks(rotation=35,ha="right")
ax.legend(["AUPRC","F1"]); annotate(ax); plt.tight_layout(); plt.savefig(OUT/"3a_performance_bars.png",dpi=130); plt.show()

fig,ax=plt.subplots(figsize=(11,7))
for i,(nm,r) in enumerate(dfL.iterrows()):
    ax.scatter(r.size_kb,r.auprc,s=240,color=PALETTE[i%8],edgecolor="k",zorder=3)
    ax.annotate(nm,(r.size_kb,r.auprc),xytext=(7,6),textcoords="offset points",fontsize=10)
ax.set_xscale("log"); ax.set_xlabel("Model size (KB, log)"); ax.set_ylabel("AUPRC")
ax.set_title("3a · Accuracy vs model size — low-resource trade-off")
plt.tight_layout(); plt.savefig(OUT/"3a_accuracy_vs_size.png",dpi=130); plt.show()

fig,ax=plt.subplots(figsize=(12,6)); ds=dfL.sort_values("infer_ms_per_1k")
ax.barh(ds.index,ds.infer_ms_per_1k,color=PALETTE[2]); ax.set_xlabel("ms per 1,000 samples")
ax.set_title("3a · Inference cost for edge deployment")
for i,v in enumerate(ds.infer_ms_per_1k): ax.text(v,i,f" {v:.1f}",va="center",fontsize=10)
plt.tight_layout(); plt.savefig(OUT/"3a_inference_latency.png",dpi=130); plt.show()
dfL.round(3)


## 3b · Robustness — cloud gaps, sensor noise, cross-year

In [ ]:
ref=clone(make_models()["RF-full (baseline)"]).fit(Xtr,ytr)
light=clone(make_models()["RF-small (50, d=8)"]).fit(Xtr,ytr)
TRAJ_te={k:v[te] for k,v in TRAJ.items()}

drops=[0,.1,.2,.3,.4,.5,.6]; cloud={"RF-full":[],"RF-small":[]}
for d in drops:
    Xc=featurize(TRAJ_te,drop_frac=d,seed=RNG)
    cloud["RF-full"].append(metrics_at(yte,proba_of(ref,Xc))["auprc"])
    cloud["RF-small"].append(metrics_at(yte,proba_of(light,Xc))["auprc"])
noises=[0,.02,.05,.08,.12,.16,.20]; nz={"RF-full":[],"RF-small":[]}
for v in noises:
    Xn=featurize(TRAJ_te,noise=v,seed=RNG)
    nz["RF-full"].append(metrics_at(yte,proba_of(ref,Xn))["auprc"])
    nz["RF-small"].append(metrics_at(yte,proba_of(light,Xn))["auprc"])
pd.DataFrame({"drop":drops,**cloud}).to_csv(OUT/"3b_cloud_sweep.csv",index=False)
pd.DataFrame({"noise":noises,**nz}).to_csv(OUT/"3b_noise_sweep.csv",index=False)

fig,ax=plt.subplots(1,2,figsize=(16,6))
for k,c in zip(cloud,[PALETTE[0],PALETTE[4]]): ax[0].plot([d*100 for d in drops],cloud[k],"o-",lw=2.5,ms=9,label=k,color=c)
ax[0].set_xlabel("% windows lost to cloud"); ax[0].set_ylabel("AUPRC"); ax[0].set_title("Cloud-gap robustness"); ax[0].legend(); ax[0].set_ylim(0,1.02)
for k,c in zip(nz,[PALETTE[0],PALETTE[4]]): ax[1].plot(noises,nz[k],"o-",lw=2.5,ms=9,label=k,color=c)
ax[1].set_xlabel("added noise (index std)"); ax[1].set_ylabel("AUPRC"); ax[1].set_title("Sensor-noise robustness"); ax[1].legend(); ax[1].set_ylim(0,1.02)
plt.tight_layout(); plt.savefig(OUT/"3b_robustness_sweeps.png",dpi=130); plt.show()

yrs=meta["year"].to_numpy(); early=yrs<=2020; late=yrs>=2021
idm=metrics_at(yte,proba_of(light,Xte))
mE=clone(make_models()["RF-small (50, d=8)"]).fit(X[early],y[early]); elm=metrics_at(y[late],proba_of(mE,X[late]))
mL=clone(make_models()["RF-small (50, d=8)"]).fit(X[late],y[late]);  lem=metrics_at(y[early],proba_of(mL,X[early]))
cy=pd.DataFrame({"setting":["In-distribution","Train ≤2020→≥2021","Train ≥2021→≤2020"],
    "AUPRC":[idm["auprc"],elm["auprc"],lem["auprc"]],"F1":[idm["f1"],elm["f1"],lem["f1"]]})
cy.to_csv(OUT/"3b_cross_year.csv",index=False)
fig,ax=plt.subplots(figsize=(10,6)); cy.set_index("setting")[["AUPRC","F1"]].plot(kind="bar",ax=ax,color=[PALETTE[1],PALETTE[5]],width=.8)
ax.set_ylim(0,1.05); ax.set_ylabel("Score"); ax.set_title("3b · Cross-year transfer (RF-small)"); plt.xticks(rotation=12); annotate(ax,"{:.2f}",10)
plt.tight_layout(); plt.savefig(OUT/"3b_cross_year.png",dpi=130); plt.show(); cy


## 3c · Cross-region generalisation (leave-one-province-out)

In [ ]:
prov=meta["province"].to_numpy(); rows=[]
for p in sorted(set(prov)):
    tem=prov==p; trm=~tem
    if tem.sum()<20 or y[tem].sum()<5: continue
    m=clone(make_models()["RF-small (50, d=8)"]).fit(X[trm],y[trm])
    rows.append(dict(province=p,n=int(tem.sum()),pos=int(y[tem].sum()),**metrics_at(y[tem],proba_of(m,X[tem]))))
dfR=pd.DataFrame(rows).sort_values("auprc",ascending=False); dfR.to_csv(OUT/"3c_cross_region.csv",index=False)
fig,ax=plt.subplots(figsize=(13,6)); xp=np.arange(len(dfR))
ax.bar(xp-.2,dfR.auprc,.4,label="AUPRC",color=PALETTE[1]); ax.bar(xp+.2,dfR.recall,.4,label="Recall",color=PALETTE[5])
ax.axhline(dfR.auprc.mean(),ls="--",c="grey",label=f"mean AUPRC={dfR.auprc.mean():.2f}")
ax.set_xticks(xp); ax.set_xticklabels(dfR.province,rotation=35,ha="right"); ax.set_ylim(0,1.05); ax.legend()
ax.set_title("3c · Leave-one-province-out (RF-small)"); plt.tight_layout(); plt.savefig(OUT/"3c_cross_region.png",dpi=130); plt.show(); dfR.round(3)


## 3d · Fairness — smallholder vs large-farm detection

In [ ]:
refm=fitted["RF-full (baseline)"]; pr=proba_of(refm,Xte); pred=(pr>=.5).astype(int)
def grec(series):
    out={}
    for g in series.unique():
        gm=(series==g).to_numpy()&(yte==1)
        if gm.sum()>=5: out[g]=dict(n_pos=int(gm.sum()),recall=float(pred[gm].mean()),mean_proba=float(pr[gm].mean()))
    return out
order=["smallholder","medium","large"]
farm=grec(meta_te["farm_class"]); farm={k:farm[k] for k in order if k in farm}
fr=pd.DataFrame(farm).T; fr.to_csv(OUT/"3d_fairness_farm_size.csv")
disp=float(fr.recall.max()-fr.recall.min())
region=grec(meta_te["province"]); region=dict(sorted(region.items(),key=lambda kv:-kv[1]["recall"]))
fig,ax=plt.subplots(1,2,figsize=(16,6)); cols=[PALETTE[0],PALETTE[3],PALETTE[6]]
ax[0].bar(fr.index,fr.recall,color=cols[:len(fr)])
for i,v in enumerate(fr.recall): ax[0].text(i,v,f"{v:.2f}",ha="center",va="bottom",fontsize=12)
ax[0].set_ylim(0,1.05); ax[0].set_ylabel("Recall (detection rate)"); ax[0].set_xlabel("Farm-size class")
ax[0].set_title(f"Detection parity by farm size\nΔrecall(large−small)={disp:.2f}")
rdf=pd.DataFrame(region).T; ax[1].bar(rdf.index,rdf.recall,color=PALETTE[4])
ax[1].axhline(rdf.recall.mean(),ls="--",c="grey",label=f"mean={rdf.recall.mean():.2f}")
ax[1].set_xticklabels(rdf.index,rotation=35,ha="right"); ax[1].set_ylim(0,1.05); ax[1].legend(); ax[1].set_title("Detection parity by province")
plt.tight_layout(); plt.savefig(OUT/"3d_fairness.png",dpi=130); plt.show()
print(f"smallholder recall={fr.loc['smallholder','recall']:.3f} vs large={fr.loc['large','recall']:.3f} (Δ={disp:.3f})"); fr.round(3)


## 3e · Explainability — feature importance + SHAP

In [ ]:
rf=fitted["RF-small (50, d=8)"]
imp=pd.Series(rf.feature_importances_,index=feat_names).sort_values(ascending=False)
imp.head(20).to_csv(OUT/"3e_feature_importance.csv")
fig,ax=plt.subplots(figsize=(11,8)); top=imp.head(15)[::-1]
ax.barh(top.index,top.values,color=PALETTE[3]); ax.set_xlabel("Gini importance")
ax.set_title("3e · RF feature importance (top 15, real S2)"); plt.tight_layout(); plt.savefig(OUT/"3e_feature_importance.png",dpi=130); plt.show()

try:
    import shap
    samp=Xte.sample(min(300,len(Xte)),random_state=RNG)
    sv=shap.TreeExplainer(rf).shap_values(samp)
    sv1=sv[1] if isinstance(sv,list) else (sv[...,1] if getattr(sv,'ndim',2)==3 else sv)
    plt.figure(); shap.summary_plot(sv1,samp,max_display=15,show=False)
    plt.tight_layout(); plt.savefig(OUT/"3e_shap_summary.png",dpi=130,bbox_inches="tight"); plt.show()
    ms=meta_te.loc[samp.index]; mab=pd.DataFrame(np.abs(sv1),columns=feat_names,index=samp.index)
    rg=[mab.loc[ms.index[ms.farm_class==g]].mean().rename(g) for g in order if (ms.farm_class==g).sum()>=5]
    if rg:
        gdf=pd.concat(rg,axis=1); tf=gdf.mean(1).sort_values(ascending=False).head(10).index
        fig,ax=plt.subplots(figsize=(12,7)); gdf.loc[tf][::-1].plot(kind="barh",ax=ax,color=cols[:gdf.shape[1]])
        ax.set_xlabel("mean |SHAP|"); ax.set_title("3e · Detection drivers by farm size")
        plt.tight_layout(); plt.savefig(OUT/"3e_shap_by_farmsize.png",dpi=130); plt.show()
except Exception as e:
    print("SHAP skipped:", e)
imp.head(15).round(4)


## Summary

All figures + CSVs are written to `outputs/phase3/`. Every number above is computed on the
**real Sentinel-2 stacks** — the reproducibility baseline plus the Phase 3 extensions
(lightweight models, robustness, cross-region generalisation, smallholder fairness, XAI).

**Key low-resource / responsible-AI findings to discuss in the write-up:**
- *Lightweight:* how much AUPRC the tiny models retain vs the full RF, at what size/latency.
- *Robustness:* AUPRC drop as cloud loss / noise increases; cross-year generalisation gap.
- *Fairness:* the smallholder-vs-large recall gap — the core equity concern for serving
  smallholder farmers, since their fields are diluted within 10 m pixels.
- *XAI:* late-dry-season NDVI/NDWI dominate, matching the agronomy of active irrigation.
